In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import os
import sys
import django
import torch

# Setup Django environment
# Adjust the path to point to the directory containing manage.py
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../hiccup_ide")))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "hiccup_ide.settings")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
django.setup()

In [3]:
from neural_data.models import WorkGraph, TempPruneSaliencyMap, Work,Model, WorkSaliencyMap
from neural_data.graph.raw import build_graph , get_parents_of_conv2d_output_coordinate
from neural_data.types import Coordinate, Conv2dOutputCoordinate, to_coord_str, Conv2dInputCoordinate, ReLUOutputCoordinate, ReLUInputCoordinate, to_coord_str_with_grid_position
from neural_data.ui_graph_types import Conv2dInputPatchNode
from neural_data.model_spec import ModelDefinition

In [4]:
model=Model.objects.first()
model_dfn = ModelDefinition.model_validate(model.definition)
coord = Conv2dOutputCoordinate(
    type="Conv2dOutputCoordinate",
    layer_type="conv2d",
    coordinate_type="output",
    layer_name="layers.2",
    channel=12,
    y=2,
    x=3,
)

In [5]:
INPUT_ALIAS = "4-Sl1"
def filter_func(coord: Coordinate):
    # only allow pos saliency map coords
    if isinstance(coord, Conv2dInputCoordinate) or isinstance(coord, ReLUInputCoordinate):
        return True
    coord_str = to_coord_str(coord)
    try:
        sm = WorkSaliencyMap.objects.get(coordinate=coord_str, input__alias=INPUT_ALIAS)
        return sm.data[coord.y][coord.x] > 0
    except WorkSaliencyMap.DosNotExist:
        raise Exception(f"Work saliency map does not exist for coordinate, query={coord_str} coord={coord}")

In [6]:
G = build_graph(
    coord, model_dfn, filter_func
)

In [7]:
for c in get_parents_of_conv2d_output_coordinate(coord, model_dfn):
    print(filter_func(c))

False
True
True
True
False
False
False
True


In [8]:
from neural_data.graph.ui_tfm import raw_to_ui_graph

ui_g = raw_to_ui_graph(G)

In [15]:
for n in ui_g.nodes():
    generic_str(n)

In [13]:
def generic_str(node):
    cn = node.__class__.__name__
    prefix = f"{cn} ({node.layer_name}) "
    if isinstance(node, Conv2dInputPatchNode):
        miny = node.patch_min_y
        minx = node.patch_min_x
        maxy = node.patch_max_y
        maxx = node.patch_max_x
        n = node
        return f"{prefix} {n.layer_name}.out_{n.out_channel}.in_{n.in_channel} ({miny},{minx}:{maxy},{maxx})"
    else:
        return f"{prefix} {to_coord_str_with_grid_position(node)}"

In [17]:
from pyvis.network import Network
net = Network(notebook=True, height="750px", width="100%", bgcolor="#222222", font_color="white", cdn_resources="remote", directed=True)
graph = ui_g
for node in graph.nodes():
    net.add_node(generic_str(node))
for (src, target) in graph.edges():
    net.add_edge(generic_str(src), generic_str(target))
net.set_options("""
{
  "layout": {
    "hierarchical": {
      "enabled": true,
      "levelSeparation": 150,
      "nodeSpacing": 500,
      "treeSpacing": 200,
      "direction": "UD",
      "sortMethod": "directed"
    }
  },
  "physics": {
    "enabled": false
  }
}
""")

net.show("graph.html")

graph.html
